In [1]:
import pandas as pd 
import numpy as np
import os
import re

from sklearn.metrics import confusion_matrix

import matplotlib.pyplot as plt  
import seaborn as sns 
sns.set(font_scale=1.4)


In [4]:
DATA_DIR = "./race_results/pos/"

files = os.listdir(DATA_DIR)
files = list(filter(lambda x: x.endswith('.csv'), files))
files

['anger-output-batch-0.csv',
 'anger-output-batch-1.csv',
 'anger-output-batch-2.csv',
 'anger-output-batch-3.csv',
 'anger-output-batch-4.csv',
 'anger-output-batch-5.csv',
 'anger-output-batch-6.csv',
 'anger-output-batch-7.csv',
 'anger-output-batch-8.csv',
 'anger-output-batch-9.csv',
 'disgust-output-batch-0.csv',
 'disgust-output-batch-1.csv',
 'disgust-output-batch-2.csv',
 'disgust-output-batch-3.csv',
 'fear-output-batch-0.csv',
 'fear-output-batch-1.csv',
 'fear-output-batch-2.csv',
 'fear-output-batch-3.csv',
 'fear-output-batch-4.csv',
 'fear-output-batch-5.csv',
 'fear-output-batch-6.csv',
 'fear-output-batch-7.csv',
 'fear-output-batch-8.csv',
 'fear-output-batch-9.csv',
 'happiness-output-batch-0.csv',
 'happiness-output-batch-1.csv',
 'happiness-output-batch-2.csv',
 'happiness-output-batch-3.csv',
 'happiness-output-batch-4.csv',
 'happiness-output-batch-5.csv',
 'sadness-output-batch-0.csv',
 'sadness-output-batch-1.csv',
 'sadness-output-batch-2.csv',
 'sadness-outpu

In [31]:
def get_label(s, logging=False):

    # assume first letter is num
    first = str(s).split()[0]
    match = -1

    frac_match = re.search(r"(\d+/\d+)", first)  # 1/5
    perc_match = re.search(r"(\d*)%", first)      # 20%
    dec_match = re.search(r"((\d+)\.(\d*))", first)  # 0.20
    num_match = re.search(r"(\d+)", first)        # 20

    if frac_match:
        if logging:
            print("FRAC MATCH")
        match = eval(frac_match[0]) * 100

    elif perc_match:
        if logging:
            print("PERCENT MATCH")
        match = float(perc_match[1])

    elif dec_match:
        if logging:
            print("DECIMAL MATCH")
        n = float(dec_match[1])
        match = n * 100 if n <= 1 else n
    
    elif num_match:
        if logging:
            print("NUM MATCH")
        n = float(num_match[1])
        match = 100 if n == 1 else n
    
    elif 'high' in first.lower():
        match = 100

    elif 'low' in first.lower():
        match = 0

    if match > -1:
        if match > 50:     # 2 is True
            return 2
        elif match == 50:  # 1 is 50/50
            return 1
        else:              # 0 is False
            return 0
        
    #print("=" * 40)
    #print("NO NUMBER FOUND")
    #print(s)
    return 0



pref_order = ['happiness', 'anger', 'sadness', 'fear', 'disgust', 'surprise']


emotion_dfs = {}

for emotion in pref_order:


    emotion_files = []
    for file in files:
        if file.startswith(emotion):
            emotion_files.append(file)

    df = pd.DataFrame()

    for file in emotion_files:
        tmp_df = pd.read_csv(DATA_DIR + file)
        df = pd.concat([df, tmp_df])

    for col in list(df.columns):
        if 'Label' in col:
            df[col] = df[col].astype(int)

    baseline_labels = df['Baseline Output'].apply(get_label)
    df['Baseline Label (Strict)'] = (baseline_labels == 2).astype('int32')
    df['Baseline Label (Loose)'] = (baseline_labels != 0).astype('int32')
    df['Baseline Label (Custom)'] = baseline_labels

    control_labels = df['Control Output'].apply(get_label)
    df['Control Label (Strict)'] = (control_labels == 2).astype('int32')
    df['Control Label (Loose)'] = (control_labels != 0).astype('int32')
    df['Control Label (Custom)'] = control_labels

    emotion_dfs[emotion] = df


In [32]:
df

,Sample Input,Sample Label,Baseline Output,Control Output,Baseline Label (Strict),Baseline Label (Loose),Baseline Label (Custom),Control Label (Strict),Control Label (Loose),Control Label (Custom)
0,Consider the correctness of the answer to the ...,0,"0.5,",0.1\n,0,1,1,0,0,0
1,Consider the correctness of the answer to the ...,0,100%,0.5\n,1,1,2,0,1,1
2,Consider the correctness of the answer to the ...,0,50%. The,"0.1,",0,1,1,0,0,0
3,Consider the correctness of the answer to the ...,0,50%.\n,0.14,0,1,1,0,0,0
4,Consider the correctness of the answer to the ...,0,"0.5,",0.1\n,0,1,1,0,0,0
...,...,...,...,...,...,...,...,...,...,...
35,Consider the correctness of the answer to the ...,1,80%. The,"0.1,",1,1,2,0,0,0
36,Consider the correctness of the answer to the ...,1,100%,"0.1,",1,1,2,0,0,0
37,Consider the correctness of the answer to the ...,1,50%. The,0.14,0,1,1,0,0,0
38,Consider the correctness of the answer to the ...,1,0.75,"0.1,",1,1,2,0,0,0


In [38]:
emotion_df['Baseline Label (Strict)'] == df['Sample Label']

Sample Input               0
Sample Label               0
Baseline Output            0
Control Output             0
Baseline Label (Strict)    0
Baseline Label (Loose)     0
Baseline Label (Custom)    0
Control Label (Strict)     0
Control Label (Loose)      0
Control Label (Custom)     0
dtype: int64

In [39]:
def print_accuracy():
        
    results = pd.DataFrame()

    for emotion_key in emotion_dfs:

        emotion_df = emotion_dfs[emotion_key]

        results = pd.concat([results, pd.DataFrame([{
            'Emotion': emotion_key,

            'Baseline Accuracy (Strict)': (emotion_df['Baseline Label (Strict)'] == emotion_df['Sample Label']).sum() / len(emotion_df),
            'Baseline Accuracy (Loose)': (emotion_df['Baseline Label (Loose)'] == emotion_df['Sample Label']).sum() / len(emotion_df),
            
            'Control Accuracy (Strict)': (emotion_df['Control Label (Strict)'] == emotion_df['Sample Label']).sum() / len(emotion_df),
            'Control Accuracy (Loose)': (emotion_df['Control Label (Loose)'] == emotion_df['Sample Label']).sum() / len(emotion_df), 
        }])])

    return results

In [40]:
print_accuracy()

,Emotion,Baseline Accuracy (Strict),Baseline Accuracy (Loose),Control Accuracy (Strict),Control Accuracy (Loose)
0,happiness,0.725,0.685,0.53,0.53
0,anger,0.635,0.62,0.59,0.59
0,sadness,0.73,0.73,0.585,0.53
0,fear,0.71,0.66,0.575,0.565
0,disgust,0.66875,0.7125,0.58125,0.58125
0,surprise,0.75,0.69375,0.49375,0.49375


In [45]:
# Calculate prediction accuracy for each emotion

pref_order = ['happiness', 'anger', 'sadness', 'fear', 'disgust', 'surprise']

results = print_accuracy()

newResults = pd.DataFrame()

for o in pref_order:
    newResults = pd.concat([newResults, results[results['Emotion'] == o]])

newResults['Baseline Accuracy (Loose)'].sum() / 6

0.6835416666666667

In [46]:
# Strict has better performance (SLIGHTLY)
(0.6835416666666667 + 0.6476851851851851) / 2

0.6656134259259259